## Imports & src definitions

All `src/` logic (UMF, User, create_dataloader, decentralized_train_n_epochs, etc.)
is inlined here so the notebook is fully self-contained.

In [34]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from torch.optim import SGD
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import networkx as nx
import time
import math
import copy
import os

torch.manual_seed(0)
np.random.seed(0)

In [35]:
from os import chdir
from pathlib import Path

new_path = Path("/Users/haowen/Documents/Decentral RS/fed-learning-main")

if new_path.exists():
    os.chdir(new_path)
    print(f"Working directory changed to: {Path.cwd()}")
else:
    print("Path does not exist.")


Working directory changed to: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [36]:
from src.models.MatrixFactorization import MF, UMF
from src.graphs import random_k_out_graph, create_graph
from src.users import User
from src.training.decentralized import (decentralized_train_n_epochs,
                                        decentralized_validate_loop)
from src.data_utils import create_batched_dataloaders, create_dataloader

In [37]:
#Make data sample iterable
from torch.utils.data import Dataset, DataLoader, TensorDataset, Sampler
import torch
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import SGD

from collections import Counter
import networkx as nx
from networkx.generators.classic import empty_graph
from networkx.utils import discrete_sequence, py_random_state, weighted_choice

import seaborn as sns
from sklearn.utils import shuffle

## Parameter Setting

In [39]:
model_type       = "umf"
val_loader_type  = "rs"
train_loader_type = "rs"
userprop         = None
n_factors        = 30
sparse           = False
batch_size       = 10
lr               = 0.03871364416669273
weight_decay     = 0.14214480688557163
mom              = 0.001
graph_seed       = 1
n_epochs         = 50
SEED             = 42

## Main

In [41]:
# ── Load sampled H&M dataset from cache ───────────────────────────────────────
# Set these to match what was used when sampling was run.
TARGET_USERS           = 1000
MIN_TRAIN_INTERACTIONS = 10
SAMPLED_DATA_DIR       = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled"

cache_tag  = f"u{TARGET_USERS}_m{MIN_TRAIN_INTERACTIONS}_s42"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

assert all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]), (
    f"Cached files not found in '{SAMPLED_DATA_DIR}/'.\n"
    "Run the hm_foaf_experiment_sampled.ipynb preprocessing section first "
    "to generate the cache."
)

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
meta_df  = pd.read_csv(meta_path)

n_users = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
n_items = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])

print(f"Loaded from cache: {cache_tag}")
print(f"Total User: {n_users}")
print(f"Total Item: {n_items}")
train_df.head()

Loaded from cache: u1000_m10_s42
Total User: 474
Total Item: 5323


,customer_id,product_code,bought
0,194,401,1
1,218,835,1
2,79,498,1
3,12,2042,1
4,344,234,2


In [10]:
# ── DataLoaders ───────────────────────────────────────────────────────────────
train_data_loader = create_dataloader(
    df=train_df, dl_type=train_loader_type, batch_size=batch_size, p=userprop
)
val_data_loader  = create_dataloader(df=val_df,  dl_type=val_loader_type)
test_data_loader = create_dataloader(df=test_df, dl_type=val_loader_type)

print(f"Train batches: {len(train_data_loader)} | "
      f"Val batches: {len(val_data_loader)} | "
      f"Test batches: {len(test_data_loader)}")

Train batches: 15483 | Val batches: 3871 | Test batches: 7214


## Parameter Tuning -- Grid Search

Run **after** cells 8-9 (data load + dataloaders). Trains every combination in `LR_GRID x WEIGHT_DECAY_GRID x MOMENTUM_GRID` (27 combos), selects the lowest best validation RMSE, and updates `lr`, `weight_decay`, `mom` in-place. Then re-run cells 10-12 (init users -> build graph -> train).


In [13]:
# ══════════════════════════════════════════════════════════════════
# Parameter Tuning -- Grid Search
#
# Calls decentralized_train_n_epochs for each (lr, wd, momentum) combo.
# Selects the combo with the lowest best validation RMSE.
# Run AFTER cells 8-9 (data load + dataloaders).
# ══════════════════════════════════════════════════════════════════
import itertools

# ── Grid ─────────────────────────────────────────────────────────
TUNE_KEY      = "random2_rs"
TUNE_EPOCHS   = 15

LR_GRID           = [0.005, 0.01,  0.05]
WEIGHT_DECAY_GRID = [0.05,  0.1,   0.3 ]
MOMENTUM_GRID     = [0.001, 0.5,   0.9 ]

param_grid = list(itertools.product(LR_GRID, WEIGHT_DECAY_GRID, MOMENTUM_GRID))

# Build tuning graph
tune_graph = random_k_out_graph(n=n_users, k=2, seed=graph_seed)

print(f"Grid search: {len(param_grid)} combinations x up to {TUNE_EPOCHS} epochs")
print(f"{'#':>3}  {'lr':>8}  {'wd':>8}  {'mom':>6}  {'best val RMSE':>14}")
print("-" * 48)

grid_results = []

for k, (lr_g, wd_g, mom_g) in enumerate(param_grid, 1):
    torch.manual_seed(0)

    # Fresh user models for this combo
    users_g = {}
    for i in range(n_users):
        m = UMF(n_items, n_factors=n_factors, sparse=sparse)
        users_g[i] = User(
            id=i, model=m,
            optimizer=SGD(m.parameters(),
                          lr=lr_g, weight_decay=wd_g, momentum=mom_g),
            model_name=model_type,
        )

    _, val_losses_g, _, _ = decentralized_train_n_epochs(
        user_models=users_g,
        train_loader=train_data_loader,
        val_loader=val_data_loader,
        epochs=TUNE_EPOCHS,
        graph=tune_graph,
    )

    best_val = min(val_losses_g)
    grid_results.append((best_val, lr_g, wd_g, mom_g))

    marker = " <--" if best_val == min(r[0] for r in grid_results) else ""
    print(f"{k:>3}  {lr_g:>8.4f}  {wd_g:>8.4f}  {mom_g:>6.3f}  "
          f"{best_val:>14.4f}{marker}")

# ── Best combo ───────────────────────────────────────────────────
best_val, best_lr, best_wd, best_mom = min(grid_results, key=lambda x: x[0])

print(f"\nBest val RMSE : {best_val:.4f}")
print(f"  lr           = {best_lr}")
print(f"  weight_decay = {best_wd}")
print(f"  momentum     = {best_mom}")

# ── Update params ────────────────────────────────────────────────
lr           = best_lr
weight_decay = best_wd
mom          = best_mom
print("\nParameters updated. Re-run cells 10-12 (init -> graph -> train).")


Grid search: 27 combinations x up to 15 epochs
  #        lr        wd     mom   best val RMSE
------------------------------------------------


  0%|          | 0/15483 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 4.1133 | Validation Loss: 3.5526 | Time Elapsed: 8.035417 sec |Commute: 30881 | Commute Cost:
2554896279

  0%|          | 0/15483 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 2.2070 | Validation Loss: 3.2003 | Time Elapsed: 7.576682 sec |Commute: 30881 | Commute Cost:
2554896279

...

Early stopping.

Total time elapsed: 120.34230087511241

 20    0.0500    0.0500   0.500          0.4886


  0%|          | 0/15483 [00:00<?, ?it/s]

ValueError: NaN loss at step 3838, user 68

In [20]:
# ── Initialise one UMF model per user ─────────────────────────────────────────
users = {}
for i in tqdm(range(n_users)):
    user_model = UMF(n_items, n_factors=n_factors, sparse=sparse)
    users[i] = User(
        id=i,
        model=user_model,
        optimizer=SGD(user_model.parameters(),
                      lr=0.0050, weight_decay=0.3000, momentum=0.900),
        model_name=model_type,
    )

  0%|          | 0/474 [00:00<?, ?it/s]

In [21]:
# ── Build graph ───────────────────────────────────────────────────────────────
graph = random_k_out_graph(n=n_users, k=2, seed=graph_seed)

In [23]:
# ── Train ─────────────────────────────────────────────────────────────────────
torch.manual_seed(0)
train_losses, val_losses, time_per_epoch, commutes = decentralized_train_n_epochs(
    user_models=users,
    train_loader=train_data_loader,
    val_loader=val_data_loader,
    epochs=n_epochs,
    graph=graph,
)

  0%|          | 0/15483 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 2.3819 | Validation Loss: 0.9295 | Time Elapsed: 7.990032 sec |Commute: 30881 | Commute Cost:
2554896279

  0%|          | 0/15483 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.6862 | Validation Loss: 0.5598 | Time Elapsed: 7.555031 sec |Commute: 30881 | Commute Cost:
2554896279

  0%|          | 0/15483 [00:00<?, ?it/s]

...

Epoch 10 | Train Loss: 0.7566 | Validation Loss: 0.4879 | Time Elapsed: 7.135972 sec |Commute: 30881 | Commute 
Cost: 2554896279

  0%|          | 0/15483 [00:00<?, ?it/s]

Epoch 11 | Train Loss: 0.7654 | Validation Loss: 0.4769 | Time Elapsed: 7.296263 sec |Commute: 30881 | Commute 
Cost: 2554896279

Early stopping.

Total time elapsed: 84.54573074984364

In [28]:
# ── Test evaluation ───────────────────────────────────────────────────────────
test_loss = decentralized_validate_loop(users, test_data_loader)

In [30]:
test_loss

0.52026623